In [1]:
from dd_solvers import *
from experiments import *
import matplotlib.pyplot as plt
import torch
import pandas as pd
import numpy as np

AMGX version 2.5.0
Built on Jul  5 2025, 09:37:04
Compiled with CUDA Runtime 12.6, using CUDA driver 12.6
The AMGX_initialize_plugins API call is deprecated and can be safely removed.


In [2]:
def add_cg_metadata(df):
    df["iterations"] = df["metadata"].map(
        lambda m: m.get("iterations") if not pd.isna(m) else None
    )
    df["cond"] = df["metadata"].map(
        lambda m: m.get("condition number estimate") if not pd.isna(m) else None
    )

## Table 1 from _A massively parallel ..._

In [3]:
dfs = []

for fine_m in range(4, 12):
    mesh_family = MeshFamily.uniform_quadrilateral(
        d=2,
        n=fine_m + 1,
    )

    factory = ExperimentFactory(
        test_case=continuous_coefficient_2d,
        mesh_family=mesh_family,
        polynomial_degree=1,
        number_of_repetitions=1,
        solution_warmup_steps=0,
        solution_measurement_steps=1,
        random_rhs=True,
    )

    for coarse_m in range(4, fine_m + 1):
        factory.add(
            Experiment(
                fine_m=fine_m + 1,
                solvers_m=fine_m + 1,
                coarse_m=coarse_m,
                solver=CG(
                    ASM(torch.float64, LU(), CUDSS()),
                    rtol=1e-6,
                    maxiter=2000,
                    bsr_matmul=False,
                    estimate_cond=True,
                ),
            )
        )

    dfs.append(factory.run())

Running experiments:   0%|          | 0/8 [00:00<?, ?it/s]W0818 18:53:12.118000 1776 torch/_dynamo/convert_frame.py:861] [4/8] torch._dynamo hit config.cache_size_limit (8)
W0818 18:53:12.118000 1776 torch/_dynamo/convert_frame.py:861] [4/8]    function: 'torch_dynamo_resume_in_solve_at_585' (/root/master-thesis/src/dd_solvers/dd_solvers/solvers.py:585)
W0818 18:53:12.118000 1776 torch/_dynamo/convert_frame.py:861] [4/8]    last reason: 4/0: tensor 'L['rhs']' size mismatch at index 0. expected 1536, actual 25165824
W0818 18:53:12.118000 1776 torch/_dynamo/convert_frame.py:861] [4/8] To log all recompilation reasons, use TORCH_LOGS="recompiles".
W0818 18:53:12.118000 1776 torch/_dynamo/convert_frame.py:861] [4/8] To diagnose recompilation issues, see https://pytorch.org/docs/main/torch.compiler_troubleshooting.html.
Running experiments: 100%|██████████| 8/8 [01:09<00:00,  8.69s/it]


In [4]:
df = pd.concat(dfs, ignore_index=True)
add_cg_metadata(df)
df["table entry"] = df.apply(
    lambda row: (
        f"{int(row['iterations'])} ({row['cond']:.2f})"
        if not pd.isna(row["iterations"])
        else "--"
    ),
    axis=1,
)
df["fine m"] = df["fine m"] - 1

In [5]:
df.pivot_table(
    values="table entry", index="coarse m", columns="fine m", aggfunc="min"
).map(lambda x: "-" if pd.isna(x) else x)

fine m,4,5,6,7,8,9,10,11
coarse m,,,,,,,,
4,31 (19.48),42 (36.57),71 (105.53),126 (352.86),236 (1311.19),445 (5028.64),853 (19748.31),1628 (77062.95)
5,-,32 (19.59),43 (36.91),71 (105.48),126 (355.87),237 (1318.81),448 (5047.29),851 (19701.88)
6,-,-,32 (19.84),43 (36.69),71 (104.75),127 (357.61),236 (1316.91),449 (5044.18)
7,-,-,-,32 (19.82),43 (36.69),71 (105.38),127 (357.34),235 (1313.83)
8,-,-,-,-,32 (19.81),43 (36.43),71 (105.43),127 (357.59)
9,-,-,-,-,-,32 (19.87),43 (36.44),71 (105.23)
10,-,-,-,-,-,-,32 (19.86),43 (36.41)
11,-,-,-,-,-,-,-,32 (19.86)


In [6]:
df.to_csv("../results/experiment_cond_h.csv", index=False)

## Dependence on $p$

In [11]:
mesh_family = MeshFamily.uniform_quadrilateral(
    d=2,
    n=4,
)

fine_m = 4
solvers_m = 3
coarse_m = 3

In [12]:
dfs = []

for p in range(1, 17):
    factory = ExperimentFactory(
        test_case=continuous_coefficient_2d,
        mesh_family=mesh_family,
        polynomial_degree=p,
        number_of_repetitions=1,
        solution_warmup_steps=0,
        solution_measurement_steps=1,
    )

    factory.add(
        Experiment(
            fine_m=fine_m,
            solvers_m=fine_m,
            coarse_m=coarse_m,
            solver=CG(
                ASM(torch.float64, LU(), CUDSS()),
                rtol=1e-6,
                maxiter=2000,
                bsr_matmul=False,
                estimate_cond=True,
            ),
        )
    )

    local_df = factory.run()
    local_df["polynomial degree"] = p
    dfs.append(local_df)

df = pd.concat(dfs, ignore_index=True)
add_cg_metadata(df)

Running experiments: 100%|██████████| 1/1 [00:02<00:00,  2.25s/it]


In [13]:
summary = df[["polynomial degree", "iterations", "cond"]]
summary["cond / p^2"] = summary["cond"] / summary["polynomial degree"] ** 2
summary

,polynomial degree,iterations,cond,cond / p^2
0,1,21,12.269330,12.269330
1,2,42,61.014122,15.253530
2,3,64,143.400024,15.933336
3,4,86,260.149968,16.259373
4,5,105,410.441387,16.417655
5,6,126,595.079059,16.529974
6,7,149,813.860361,16.609395
7,8,168,1065.693645,16.651463
8,9,189,1351.682010,16.687432
9,10,222,1684.177577,16.841776


In [14]:
df.to_csv("../results/experiment_cond_p.csv", index=False)